In [1]:
cd ..

/home/jovyan/Robbi/dea-intertidal


In [ ]:
pip install -e .

In [ ]:
pip install eo-tides

In [ ]:
pip install dea-tools

In [2]:
import os
import sys
import numpy as np
import click
import xarray
import datacube
import odc.geo.xr
from odc.geo.geom import BoundingBox
from odc.algo import (
    int_geomedian,
    keep_good_only,
    xr_quantile,
)
from datacube.utils.aws import configure_s3_access
from eo_tides.eo import pixel_tides
from dea_tools.dask import create_local_dask_cluster

from intertidal.utils import configure_logging
from intertidal.io import (
    load_data,
    prepare_for_export,
    tidal_metadata,
    export_dataset_metadata,
)

from ipyleaflet import basemaps, basemap_to_tiles
from odc.geo.geobox import GeoBox
from odc.ui import select_on_a_map

In [ ]:
# Set study area name for outputs
study_area = "testing"

# Plot interactive map to select area
basemap = basemap_to_tiles(basemaps.Esri.WorldImagery)
geom = select_on_a_map(height="600px", layers=(basemap,), center=(-26, 135), zoom=4)
geom

In [3]:
# Set study area (e.g. tile ID in form "x094y145")
study_area = "x201y141"
geom = None  # Use GridSpec to load study area, not a custom geom

In [39]:
# Create local dask cluster to improve data load time
client = create_local_dask_cluster(return_client=True)

# Connect to datacube to load data
dc = datacube.Datacube(app="Composites_CLI")

def filter_granules(dataset):
    """
    Return False for any Sentinel-2 dataset with a MGRS
    granule region code in the list of bad region codes.
    """
    drop_list = ["50HKG", "50HNF", "51LWD", "51LXE", "51LZF",
                 "52LBL", "52LCL", "52LDK", "53HNA", "53LRC",
                 "54GYU", "54LWR", "54LXR", "54LYR", "55GBP",
                 "55KEA", "55KFV", "55KGV", "55KHT", "55KHU",
                 "56KKC", "56KLC", "56KMC", "56KMV", "56KNU",
                 "54LWQ", "54LWP"]
    return dataset.metadata.region_code not in drop_list


# Load satellite data and dataset IDs for metadata
satellite_ds, dss_s2, dss_ls = load_data(
    dc=dc,
    study_area=study_area,
    geom=geom,
    time_range=("2021", "2023"),
    resolution=10,
    crs="EPSG:3577",
    include_s2=True,
    include_ls=False,
    filter_gqa=False,
    ndwi=False,
    mask_sunglint=20,
    include_coastal_aerosol=True,
    max_cloudcover=90,
    skip_broken_datasets=True,
    dataset_maturity="final",
    dtype="int16",
    dataset_predicate=filter_granules,
)

satellite_ds

/env/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39589 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/robbi.bishoptaylor@ga.gov.au/proxy/39589/status,
Dashboard: /user/robbi.bishoptaylor@ga.gov.au/proxy/39589/status,Workers: 1
Total threads: 15,Total memory: 117.21 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37385,Workers: 1
Dashboard: /user/robbi.bishoptaylor@ga.gov.au/proxy/39589/status,Total threads: 15
Started: Just now,Total memory: 117.21 GiB
Comm: tcp://127.0.0.1:45369,Total threads: 15
Dashboard: /user/robbi.bishoptaylor@ga.gov.au/proxy/42197/status,Memory: 117.21 GiB
Nanny: tcp://127.0.0.1:46675,


<xarray.Dataset> Size: 18GB
Dimensions:                (time: 80, y: 3200, x: 3200)
Coordinates:
  * time                   (time) datetime64[ns] 640B 2021-01-12T00:12:39.650...
  * y                      (y) float64 26kB -2.368e+06 -2.368e+06 ... -2.4e+06
  * x                      (x) float64 26kB 2.016e+06 2.016e+06 ... 2.048e+06
    spatial_ref            int32 4B 3577
Data variables:
    nbart_blue             (time, y, x) int16 2GB dask.array<chunksize=(1, 3200, 3200), meta=np.ndarray>
    nbart_green            (time, y, x) int16 2GB dask.array<chunksize=(1, 3200, 3200), meta=np.ndarray>
    nbart_red              (time, y, x) int16 2GB dask.array<chunksize=(1, 3200, 3200), meta=np.ndarray>
    nbart_red_edge_1       (time, y, x) int16 2GB dask.array<chunksize=(1, 3200, 3200), meta=np.ndarray>
    nbart_red_edge_2       (time, y, x) int16 2GB dask.array<chunksize=(1, 3200, 3200), meta=np.ndarray>
    nbart_red_edge_3       (time, y, x) int16 2GB dask.array<chunksize=(1, 3200, 3200), meta=np.ndarray>
    nbart_nir_1            (time, y, x) int16 2GB dask.array<chunksize=(1, 3200, 3200), meta=np.ndarray>
    nbart_nir_2            (time, y, x) int16 2GB dask.array<chunksize=(1, 3200, 3200), meta=np.ndarray>
    nbart_swir_2           (time, y, x) int16 2GB dask.array<chunksize=(1, 3200, 3200), meta=np.ndarray>
    nbart_swir_3           (time, y, x) int16 2GB dask.array<chunksize=(1, 3200, 3200), meta=np.ndarray>
    nbart_coastal_aerosol  (time, y, x) int16 2GB dask.array<chunksize=(1, 3200, 3200), meta=np.ndarray>
Attributes:
    crs:           EPSG:3577
    grid_mapping:  spatial_ref

In [38]:
# dss_s2[2].metadata.fields

In [34]:
[i.metadata.region_code for i in dss_s2]

['56KLB',
 '56KLC',
 '56KLC',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLC',
 '56KLC',
 '56KLC',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLC',
 '56KLC',
 '56KLC',
 '56KLC',
 '56KLC',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLC',
 '56KLC',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB',
 '56KLB']

In [29]:
filter_granules(dss_s2[140])

IndexError: list index out of range

In [ ]:
# Model tides into for spatial extent and timesteps in satellite data
tides_highres = pixel_tides(
    data=satellite_ds,
    model="ensemble",
    resample=True,
    directory="/var/share/tide_models/",
)

# Identify nodata pixels in satellite data array by loading only
# a single band into memory
nodata = satellite_ds.nbart_red.nodata
nodata_array = (satellite_ds.nbart_red != nodata).compute()

# Mask tides to make nodata match satellite data array
tides_highres = tides_highres.where(nodata_array)

In [ ]:
def tidal_thresholds(
    tides_highres,
    threshold_lowtide=0.15,
    threshold_hightide=0.85,
    min_obs=0,
):
    # Calculate per-pixel integer rankings for each tide height
    rank_n = tides_highres.rank(dim="time")

    # Calculate pixel-based low and high ranking thresholds from
    # max ranking. Max ranking needs to be rounded up to the nearest
    # integer using "ceil" as xarray will give multiple observation
    # an average rank (e.g. 50.5) value if they are both identical.
    # Additionally: to ensure we capture all matching values, Low
    # threshold needs to be rounded up ("ceil"), and high tide
    # rounded down ("floor").
    rank_max = np.ceil(rank_n.max(dim="time"))
    rank_thresh_low = np.ceil(rank_max * threshold_lowtide)
    rank_thresh_high = np.floor(rank_max * threshold_hightide)

    # Update thresholds to ensure minimum number of valid observations
    if min_obs > 0:
        rank_thresh_low = np.maximum(rank_thresh_low, min_obs)
        rank_thresh_high = np.minimum(rank_thresh_high, rank_max - min_obs)

    # Calculate tide thresholds by masking tides by ranking threshold
    tide_thresh_low = tides_highres.where(rank_n <= rank_thresh_low).max(dim="time")
    tide_thresh_high = tides_highres.where(rank_n >= rank_thresh_high).min(dim="time")

    return tide_thresh_low, tide_thresh_high

In [ ]:
cpus = None
max_iters = 10
eps = 1e-4
threshold_lowtide=0.15
threshold_hightide=0.85
min_obs=20

low_threshold, high_threshold = tidal_thresholds(
    tides_highres=tides_highres,
    threshold_lowtide=threshold_lowtide,
    threshold_hightide=threshold_hightide,
    min_obs=min_obs,
)

# Create masks for selecting satellite observations below and above the
# low and high tide thresholds
low_mask = tides_highres <= low_threshold
high_mask = tides_highres >= high_threshold

# # Keep only scenes with at least some valid data to speed up geomedian
# low_keep = low_mask.any(dim=["x", "y"])
# high_keep = high_mask.any(dim=["x", "y"])
# ds_low = satellite_ds.sel(time=low_keep)
# ds_high = satellite_ds.sel(time=high_keep)

# # Load low and high subsets of data into memory
# ds_low.load()
# ds_high.load()

# # Use `keep_good_only` to set any pixels outside of the tide masks to nodata
# ds_low_masked = keep_good_only(x=ds_low, where=low_mask.sel(time=low_keep))
# ds_high_masked = keep_good_only(x=ds_high, where=high_mask.sel(time=high_keep))

# # Calculate low and high tide geomedians
# num_threads = cpus if cpus is not None else os.cpu_count() - 2
# ds_lowtide = int_geomedian(
#     ds=ds_low_masked,
#     maxiters=max_iters,
#     num_threads=num_threads,
#     eps=eps,
# )
# ds_hightide = int_geomedian(
#     ds=ds_high_masked,
#     maxiters=max_iters,
#     num_threads=num_threads,
#     eps=eps,
# )

# # Calculate clear count (both low and high tide clear counts
# # are identical, so we can just use one)
# ds_lowtide["qa_count_clear"] = (
#     (ds_low_masked.nbart_red != nodata).sum(dim="time").astype("int16")
# )

# # Add low and high tide thresholds to the output datasets
# ds_lowtide["qa_low_threshold"] = low_threshold
# ds_hightide["qa_high_threshold"] = high_threshold


In [ ]:
low_mask.mean(dim=["x", "y"]).plot()

In [ ]:
# CURRENT LOGIC: Loading any image with even a single low tide pixel
low_mask.any(dim=["x", "y"]).sum().item()

In [ ]:
# Loading only images with more than 1% low tide pixels
(low_mask.mean(dim=["x", "y"]) > 0.01).sum().item()

In [ ]:
# Loading only images with more than 5% low tide pixels
(low_mask.mean(dim=["x", "y"]) > 0.05).sum().item()

In [ ]:
import xarray as xr

# Add new array to data to label high or low tide
# images selected for geomedian analysis.
label = xr.where(
    high_keep | low_keep,
    "Clear low and high tide images",
    "All satellite observations",
)
satellite_ds = satellite_ds.assign(label=label)

In [ ]:
tides_1d = tides_highres.median(dim=["x", "y"])
tides_1d_q = tides_1d.quantile(q=[0.15, 0.85])
low_keep = tides_1d < tides_1d_q.isel(quantile=0)
high_keep = tides_1d > tides_1d_q.isel(quantile=1)

# Add new array to data to label high or low tide
# images selected for geomedian analysis.
label = xr.where(
    high_keep | low_keep,
    "Clear low and high tide images",
    "All satellite observations",
)
satellite_ds = satellite_ds.assign(label=label)

In [ ]:
from intertidal.io import tidal_metadata

model="EOT20"
directory="/var/share/tide_models/"

# Calculate additional tile-level tidal metadata and graph.
metadata_dict, tide_graph_fig = tidal_metadata(
    product_family="tidal_composites",
    data=satellite_ds,
    plot_var="label",
    modelled_freq="30min",
    model=model,
    directory=directory,
)


In [ ]:
from eo_tides.stats import tide_stats
import matplotlib.pyplot as plt


def tidal_metadata(
    product_family,
    threshold_lowtide=0.15,
    threshold_hightide=0.85,
    **tide_stats_kwargs,
):
    """
    Generate tidal statistics and tide bias plot for a given input tile.
    Tidal statistics are calculated based on the centroid of the tile.

    For `product_family=="tidal_composites"`, observations matching the
    low and high tide thresholds will be plotted in white.

    Parameters
    ----------
    product_family : string
        Either "intertidal" or "tidal_composites".
    threshold_lowtide : float, optional
        Quantile used to identify low tide observations, by default 0.15.
    threshold_hightide : float, optional
        Quantile used to identify high tide observations, by default 0.85.
    **tide_stats_kwargs :
        Any required parameters to pass to `eo_tides.stats.tide_stats`,
        e.g. `data`, `model`, `directory` etc.

    Returns
    -------
    metadata_dict : dict
        A dictionary of tidal statistics for the tile.
    fig : matplotlib.figure.Figure
        A matplotlib figure depicting observed and modelled tides.
    """

    # Run tidal stats based on centre of tile
    metadata_df = tide_stats(
        plain_english=False,
        **tide_stats_kwargs,
    )
    fig = plt.gcf()

    # Update to use expected metadata format and rounding
    metadata_dict = (
        metadata_df.drop(["mot", "mat", "x", "y"]).add_prefix("intertidal:").to_dict()
    )
    metadata_dict = {k: round(v, 3) for k, v in metadata_dict.items()}

    # Calculate macro/meso/micro-tidal category
    metadata_dict["intertidal:tr_class"] = (
        "microtidal"
        if metadata_dict["intertidal:tr"] < 2
        else (
            "mesotidal"
            if 2 <= metadata_dict["intertidal:tr"] <= 4
            else "macrotidal" if metadata_dict["intertidal:tr"] > 4 else np.nan
        )
    )

    # Update figure line and point colours
    modelled = fig.axes[0].get_lines()[0]
    observed = fig.axes[0].get_lines()[1]
    hat = fig.axes[0].get_lines()[2]
    hot = fig.axes[0].get_lines()[3]
    lot = fig.axes[0].get_lines()[4]
    lat = fig.axes[0].get_lines()[5]

    # Set styling
    modelled.set_color("#90b7d8")
    modelled.set_alpha(1.0)
    observed.set_color("black")
    observed.set_markersize(4)
    observed.set_markeredgecolor("none")

    # Remove HAT/LOT lines
    hat.set_color("none")
    hot.set_color("none")
    lot.set_color("none")
    lat.set_color("none")

    # For Tidal Composites, manually set low and high
    # tide observations to white
    if product_family == "tidal_composites":

        # Extract observed data from plot
        xdata = observed.get_xdata()
        ydata = observed.get_ydata()

        # Calculate thresholds and plot subset of points in white
        min_thresh, max_thresh = np.quantile(
            ydata, [threshold_lowtide, threshold_hightide]
        )
        mask = (ydata <= min_thresh) | (ydata >= max_thresh)
        fig.axes[0].plot(
            xdata[mask],
            ydata[mask],
            marker="o",
            linestyle="None",
            color="white",
            markersize=5,
            markeredgecolor="#343c47",
            markeredgewidth=0.8,
            label="Low and high tide images",
        )

    # Set background to transparent
    fig.patch.set_facecolor("#5d646c00")
    fig.axes[0].set_facecolor("#5d646c00")

    # Set spines and axis labels to white
    for spine in fig.axes[0].spines.values():
        spine.set_edgecolor("#ffffff")
    fig.axes[0].tick_params(axis="both", colors="#ffffff")
    fig.axes[0].yaxis.label.set_color("#ffffff")

    # Update the legend
    legend = fig.axes[0].get_legend()
    legend.remove()
    fig.axes[0].legend(
        loc="upper center",
        bbox_to_anchor=(0.5, 1.11),
        ncol=20,
        borderaxespad=0,
        frameon=False,
        labelcolor="white",
    )

    fig.set_size_inches(8, 2.5)
    return metadata_dict, fig


model = "ensemble"
directory = "/var/share/tide_models/"

# Calculate additional tile-level tidal metadata and graph.
metadata_dict, tide_graph_fig = tidal_metadata(
    product_family="tidal_composites",
    # product_family="intertidal",
    threshold_lowtide=0.15,
    threshold_hightide=0.85,
    data=satellite_ds,
    modelled_freq="30min",
    model=model,
    directory=directory,
)

tide_graph_fig.savefig("testing.png", bbox_inches="tight")

In [ ]:
tide_stats_kwargs = dict(data=satellite_ds,
                         modelled_freq="30min",
                         model=model,
                         directory=directory)


# Run tidal stats based on centre of tile
metadata_df = tide_stats(
    plain_english=False,
    **tide_stats_kwargs,
)
fig = plt.gcf()

# Update to use expected metadata format and rounding
metadata_dict = (
    metadata_df.drop(["mot", "mat", "x", "y"]).add_prefix("intertidal:").to_dict()
)
metadata_dict = {k: round(v, 3) for k, v in metadata_dict.items()}

# Calculate macro/meso/micro-tidal category
metadata_dict["intertidal:tr_class"] = (
    "microtidal"
    if metadata_dict["intertidal:tr"] < 2
    else (
        "mesotidal"
        if 2 <= metadata_dict["intertidal:tr"] <= 4
        else "macrotidal" if metadata_dict["intertidal:tr"] > 4 else np.nan
    )
)

In [ ]:
modelled = fig.axes[0].get_lines()[0]
observed = fig.axes[0].get_lines()[1]
hat = fig.axes[0].get_lines()[2]
hot = fig.axes[0].get_lines()[3]
lot = fig.axes[0].get_lines()[4]
lat = fig.axes[0].get_lines()[5]

modelled.set_color("#90b7d8")
modelled.set_alpha(1.0)
observed.set_markersize(4)
observed.set_markeredgecolor("none")

# Set values outside range to white
xdata = observed.get_xdata()
ydata = observed.get_ydata()
min_thresh, max_thresh = np.quantile(ydata, [0.15, 0.85])
mask = (ydata <= min_thresh) | (ydata >= max_thresh)
fig.axes[0].plot(
    xdata[mask],
    ydata[mask],
    marker="o",
    linestyle="None",
    color="white",
    markersize=5,
    label="Low and high tide observations",
)

# HAT/LOT lines
hat.set_color("none")
hot.set_color("none")
lot.set_color("none")
lat.set_color("none")

# Set background to transparent
fig.patch.set_facecolor("#5d646c00")
fig.axes[0].set_facecolor("#5d646c00")

# Set spines and axis labels to white
for spine in fig.axes[0].spines.values():
    spine.set_edgecolor("#ffffff")
fig.axes[0].tick_params(axis="both", colors="#ffffff")
fig.axes[0].yaxis.label.set_color("#ffffff")

# Update the legend
legend = fig.axes[0].get_legend()
legend.remove()
fig.axes[0].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.11),
    ncol=20,
    borderaxespad=0,
    frameon=False,
    labelcolor="white",
)

fig

In [ ]:
    # Update figure line and point colours
    if product_family == "intertidal":
        fig.axes[0].get_lines()[0].set_color("#90b7d8")
        fig.axes[0].get_lines()[0].set_alpha(1.0)
        fig.axes[0].get_lines()[1].set_color("black")
        fig.axes[0].get_lines()[1].set_markersize(4)
        fig.axes[0].get_lines()[1].set_markeredgecolor("none")

        # HAT/LOT lines
        fig.axes[0].get_lines()[2].set_color("none")
        fig.axes[0].get_lines()[3].set_color("none")
        fig.axes[0].get_lines()[4].set_color("none")
        fig.axes[0].get_lines()[5].set_color("none")

    elif product_family == "tidal_composites":
        modelled = fig.axes[0].get_lines()[0]
        observed_all = fig.axes[0].get_lines()[1]

        
        modelled_tides.set_color("#90b7d8")
        modelled_tides.set_alpha(1.0)
        fig.axes[0].get_lines()[1].set_markeredgecolor("none")
        fig.axes[0].get_lines()[2].set_markeredgecolor("#343c47")
        fig.axes[0].get_lines()[1].set_marker("o")
        fig.axes[0].get_lines()[2].set_marker("o")
        fig.axes[0].get_lines()[1].set_markersize(4)
        fig.axes[0].get_lines()[2].set_markersize(5)
        fig.axes[0].get_lines()[1].set_color("black")
        fig.axes[0].get_lines()[2].set_color("white")

        # HAT/LOT lines
        fig.axes[0].get_lines()[3].set_color("none")
        fig.axes[0].get_lines()[4].set_color("none")
        fig.axes[0].get_lines()[5].set_color("none")
        fig.axes[0].get_lines()[6].set_color("none")

    # Set background to transparent
    fig.patch.set_facecolor("#5d646c00")
    fig.axes[0].set_facecolor("#5d646c00")

    # Set spines and axis labels to white
    for spine in fig.axes[0].spines.values():
        spine.set_edgecolor("#ffffff")
    fig.axes[0].tick_params(axis="both", colors="#ffffff")
    fig.axes[0].yaxis.label.set_color("#ffffff")

    # Update the legend
    legend = fig.axes[0].get_legend()
    legend.remove()
    fig.axes[0].legend(
        loc="upper center",
        bbox_to_anchor=(0.5, 1.11),
        ncol=20,
        borderaxespad=0,
        frameon=False,
        labelcolor="white",
    )

    fig.set_size_inches(8, 2.5)

In [ ]:
fig.axes[0].get_lines()[0].get_xdata()
# ydata = line.get_ydata()

In [ ]:
plt.show(tide_graph_fig)

In [ ]:
# ds_lowtide.odc.explore(bands=["nbart_red", "nbart_green", "nbart_blue"], vmin=10, vmax=2000)

## Mosaics

In [ ]:
import os 

os.system(
    f"dea-mosaics "
    f"--product ga_s2_tidal_composites_cyear_3 "
    f"--version 0-0-2 "
    f"--year 2022 "
    f"--band low-count-clear "
    f"--dataset_maturity final "
    f"--product_dir s3://dea-public-data-dev/derivative/ "
    f"--output_dir /gdata1/scratch/ "
    f"--compress ZSTD "
    f"--overview_count 8"
)

In [ ]:
!gdalinfo /gdata1/scratch/ga_s2_tidal_composites_cyear_3/0-0-2/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_low-nir-1_zstd_7overviews.tif

In [ ]:
!mv /gdata1/scratch/ga_s2_tidal_composites_cyear_3/0-0-2/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_low-nir-1.tif /gdata1/scratch/ga_s2_tidal_composites_cyear_3/0-0-2/continental_mosaics/2022--P1Y/ga_s2_tidal_composites_cyear_3_2022_low-nir-1_zstd_8overviews.tif

In [ ]:
import eo_tides